# 01b — Single Molecule Debug

Deep-dive into one molecule at a time.  Useful for:
- Verifying that a specific molecule's graph is constructed correctly
- Checking RDKit → PyG featurization manually
- Visualising 3D geometry (bond lengths, angle triplets, torsion quads)
- Catching edge cases (single-atom, disconnected, all-aromatic)

Change `MOL_IDX` to inspect any molecule in the dataset.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if Path.cwd().name == 'IG-MPNN' else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from src.data import QM9Dataset
from src.data.featurizer import ATOM_TYPES

In [ ]:
MOL_IDX = 42   # ← change this to inspect a different molecule

ds = QM9Dataset(root=ROOT / 'data', target_idx=0)
d  = ds[MOL_IDX]
print(f'Molecule index: {MOL_IDX}')
print(d)

## Node features — per atom

In [ ]:
feature_names = [
    'Z/9', 'is_H', 'is_C', 'is_N', 'is_O', 'is_F',
    'charge', 'impl_Hs/4', 'in_ring', 'aromatic', 'degree/4',
]
element_map = {0: 'H', 1: 'H', 2: 'C', 3: 'N', 4: 'O', 5: 'F'}  # one-hot col → symbol

x = d.x
print(f'Node feature tensor: {x.shape}')
print()
header = f'{"Atom":<6}' + ''.join(f'{n:>12}' for n in feature_names)
print(header)
print('-' * len(header))

for atom_idx in range(x.shape[0]):
    row = x[atom_idx]
    # Determine element from one-hot cols 1–5
    one_hot = row[1:6].tolist()
    elem = ['H','C','N','O','F'][one_hot.index(max(one_hot))] if max(one_hot) > 0 else '?'
    vals = ''.join(f'{v.item():>12.4f}' for v in row)
    print(f'{elem:<6}{vals}')

## Bond features — per directed edge

In [ ]:
bond_names = ['single', 'double', 'triple', 'aromatic']
src, dst = d.edge_index[0].tolist(), d.edge_index[1].tolist()
ea = d.edge_attr
dist = d.edge_attr_geo if hasattr(d, 'edge_attr_geo') else None

print(f'Edge feature tensor: {ea.shape}')
print(f'{"Edge":<12} {"Type":<12} {"Length (Å)"}' if dist is not None else f'{"Edge":<12} {"Type":<12}')
print('-' * 40)

for i, (s, t) in enumerate(zip(src, dst)):
    btype = bond_names[ea[i].argmax().item()]
    length_str = f'{dist[i].item():.4f}' if dist is not None else ''
    print(f'{s}→{t:<9} {btype:<12} {length_str}')

## 3D coordinates and geometry

In [ ]:
pos = d.pos.numpy()
print(f'3D coordinates (Å):')
print(f'{"Atom":<6} {"X":>8} {"Y":>8} {"Z":>8}')
print('-' * 34)
for i, (x_, y_, z_) in enumerate(pos):
    print(f'{i:<6} {x_:>8.4f} {y_:>8.4f} {z_:>8.4f}')

In [ ]:
if hasattr(d, 'angle_attr') and d.angle_attr.shape[0] > 0:
    import math
    ai = d.angle_index
    aa = d.angle_attr
    print(f'Bond angle triplets: {aa.shape[0]}')
    print(f'{"i–j–k":<12} {"Angle (°)":>10}')
    print('-' * 24)
    for k in range(min(15, aa.shape[0])):
        i, j, l = ai[0,k].item(), ai[1,k].item(), ai[2,k].item()
        deg = math.degrees(aa[k].item())
        print(f'{i}–{j}–{l:<8} {deg:>10.2f}')
    if aa.shape[0] > 15:
        print(f'  ... ({aa.shape[0] - 15} more)')
else:
    print('No angle data.')

In [ ]:
if hasattr(d, 'torsion_attr') and d.torsion_attr.shape[0] > 0:
    import math
    ti = d.torsion_index
    ta = d.torsion_attr
    print(f'Torsion angle quadruplets: {ta.shape[0]}')
    print(f'{"i–j–k–l":<16} {"Torsion (°)":>12}')
    print('-' * 30)
    for k in range(min(15, ta.shape[0])):
        i, j, l, m = ti[0,k].item(), ti[1,k].item(), ti[2,k].item(), ti[3,k].item()
        deg = math.degrees(ta[k].item())
        print(f'{i}–{j}–{l}–{m:<10} {deg:>12.2f}')
    if ta.shape[0] > 15:
        print(f'  ... ({ta.shape[0] - 15} more)')
else:
    print('No torsion data.')

## 3D scatter plot of atoms

In [ ]:
x_feat = d.x
one_hot = x_feat[:, 1:6]  # H C N O F
elem_idx = one_hot.argmax(dim=1).tolist()
elem_colors = {0: '#aec6cf', 1: '#3a7ebf', 2: '#4caf50', 3: '#e53935', 4: '#9c27b0'}
elem_names  = {0: 'H', 1: 'C', 2: 'N', 3: 'O', 4: 'F'}
colors = [elem_colors[e] for e in elem_idx]

fig = plt.figure(figsize=(7, 6))
ax  = fig.add_subplot(111, projection='3d')

ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2], c=colors, s=120, depthshade=True, zorder=5)

# Draw bonds
for s, t in zip(src, dst):
    if s < t:   # avoid drawing each bond twice
        xs = [pos[s, 0], pos[t, 0]]
        ys = [pos[s, 1], pos[t, 1]]
        zs = [pos[s, 2], pos[t, 2]]
        ax.plot(xs, ys, zs, 'k-', linewidth=1.0, alpha=0.5)

# Label atoms
for i, (x_, y_, z_) in enumerate(pos):
    ax.text(x_, y_, z_, f' {elem_names.get(elem_idx[i], "?")}{i}', fontsize=7)

ax.set_title(f'Molecule {MOL_IDX}  —  3D atom positions')
ax.set_xlabel('X (Å)')
ax.set_ylabel('Y (Å)')
ax.set_zlabel('Z (Å)')
plt.tight_layout()
plt.show()

## Adjacency matrix

In [ ]:
import torch
N = d.x.shape[0]
adj = torch.zeros((N, N))
for s, t in zip(d.edge_index[0].tolist(), d.edge_index[1].tolist()):
    adj[s, t] = 1.0

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(adj.numpy(), cmap='Blues', vmin=0, vmax=1)
ax.set_title(f'Adjacency matrix — mol {MOL_IDX}  ({N} atoms)')
ax.set_xlabel('Destination atom')
ax.set_ylabel('Source atom')
ax.set_xticks(range(N))
ax.set_yticks(range(N))
plt.tight_layout()
plt.show()